# 03b — Multi-seed evaluation (with deltas)

**Parallel sensitivity.** Same 5 seeds (`0…4`), same `eval_core.py`, extra ten `Delta *` columns. Optional `--seeds 0-19` later if requested.

Loads `data/modelling_landmark_with_deltas_trees.csv`. Promotion to the paper model only if mean test 3-class macro F1 is ≥ 0.02 higher than notebook 03 **and** Wilcoxon p < 0.05 (`evaluation_protocol.md` §9).

```bash
python revision_work/eval_core.py --which with_deltas --seeds 0-4
```

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'revision_work':
    REV = ROOT
    ROOT = ROOT.parent
else:
    REV = ROOT / 'revision_work'
if str(REV) not in sys.path:
    sys.path.insert(0, str(REV))

from eval_core import (
    DELTA_COLS, SEEDS, compare_delta_promotion, require_boosting,
    run_evaluation, tree_feature_columns,
)

require_boosting()
CSV = REV / 'data' / 'modelling_landmark_with_deltas_trees.csv'
OUT = REV / 'results' / 'with_deltas'
NO_DELTA = REV / 'results' / 'no_deltas'
df = pd.read_csv(CSV)
print(df.shape)
print(df['falls_class'].value_counts().sort_index().to_dict())
assert df['PATNO'].is_unique
for c in tree_feature_columns(use_deltas=True):
    assert c in df.columns, c
assert all(c in df.columns for c in DELTA_COLS)
print('predictors', len(tree_feature_columns(True)))
print('out', OUT)

## 5-seed run

In [ ]:
rows, summary = run_evaluation(
    CSV, OUT, use_deltas=True, seeds=SEEDS, resume=True,
)
print(json.dumps(summary, indent=2))

## Headline + pre-specified promotion vs no-deltas

In [ ]:
seeds = pd.read_csv(OUT / 'seeds.csv')
display(seeds)

def _fmt(d):
    return f"{d['mean']:.3f} (SD {d['sd']:.3f}; 95% CI {d['ci_low']:.3f}–{d['ci_high']:.3f})"

print('Two-stage (deltas) test macro F1', _fmt(summary['two_stage_f1_macro']))
print('Direct    (deltas) test macro F1', _fmt(summary['direct_f1_macro']))
w = summary['two_stage_minus_direct']
print(f"Two-stage − direct: mean {w['mean_diff']:.3f}, Wilcoxon p={w['p_value']:.4f}")

no_path = NO_DELTA / 'summary.json'
if not no_path.exists():
    print('Notebook 03 has not finished; skip promotion test.')
else:
    no_rows = [json.loads((NO_DELTA / f'seed_{s}.json').read_text()) for s in seeds['seed']]
    promo = compare_delta_promotion(no_rows, rows)
    print(json.dumps(promo, indent=2))
    print('Promote deltas to paper model:', promo['promote_deltas_to_paper_model'])